In [ ]:
%%capture
!pip install unsloth
!pip install transformers datasets accelerate bitsandbytes peft trl
!pip install scikit-learn

In [ ]:
import json
import pandas as pd
import warnings
import logging

# Suppress all the noisy warnings
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

from google.colab import drive
drive.mount('/content/drive')

filename = "/content/drive/MyDrive/Colab Notebooks/out_features.jsonl"

data = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total rows: {len(df)}")
print(df['label'].value_counts())

Mounted at /content/drive
Total rows: 80000
label
phishing    40000
benign      40000
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

phishing = df[df['label'] == 'phishing'].sample(n=5000, random_state=42)
benign = df[df['label'] == 'benign'].sample(n=5000, random_state=42)

subset = pd.concat([phishing, benign]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(
    subset, test_size=2000, random_state=42, stratify=subset['label']
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
print("Train labels:", train_df['label'].value_counts().to_dict())
print("Test labels:", test_df['label'].value_counts().to_dict())

Train: 8000  |  Test: 2000
Train labels: {'benign': 4000, 'phishing': 4000}
Test labels: {'benign': 1000, 'phishing': 1000}


In [ ]:
SYSTEM_PROMPT = """You are a web security analyst specialized in detecting phishing websites.

You will receive structured information about a website, including:
- The URL and final URL (after redirects)
- Page title and visible text
- Measurements (URL length, form counts, password inputs, anchor stats, etc.)
- Detected signals with direction (suspicious/safe/uncertain), severity (low/medium/high), and a statement

Your task:
1. Analyze ALL evidence — both suspicious AND safe signals.
2. High-severity suspicious signals (credential forms on unknown domains, brand-domain mismatches) are strong phishing indicators.
3. Safe signals (matching contact domains, functional navigation, internal links) are strong benign indicators.
4. When signals conflict, weigh severity: a high-severity suspicious signal outweighs multiple low-severity safe signals.
5. When NO signals are present, rely on URL structure and measurements.
6. Provide a specific explanation (2-4 sentences) referencing the actual evidence you see.
7. Output a final prediction.

Respond in this exact format:
Explanation: <your reasoning citing specific evidence>
Prediction: <phishing or benign>
"""

In [ ]:
def format_input(row):
    """Turn a dataframe row into a compact text description for the model."""
    inp = row['input']
    meas = row['measurements']
    feats = row['features']

    text = f"URL: {inp.get('url', '')}\n"
    text += f"Final URL: {inp.get('final_url', '')}\n"
    text += f"Title: {inp.get('title', '')}\n"

    visible = inp.get('visible_text', '') or ''
    if len(visible) > 400:
        visible = visible[:400] + "..."
    text += f"Visible text: {visible}\n\n"

    text += "Measurements:\n"
    for k, v in meas.items():
        text += f"  - {k}: {v}\n"

    text += "\nDetected signals:\n"
    if not feats:
        text += "  (none)\n"
    else:
        for f in feats:
            text += (
                f"  - [{f.get('severity','?')}/{f.get('direction','?')}] "
                f"{f.get('id','?')}: {f.get('statement','')}\n"
            )

    return text

# Preview
print(format_input(train_df.iloc[0]))

URL: https://www.dacomacoop.com/pages/contact.php
Final URL: https://www.dacomacoop.com/pages/contact.php
Title: Dacoma Farmers Coop - Contact Us
Visible text: Home | About Us | Contact Us | Hours | Locations | Pictures | Patron Login | Admin Login Cash Bids Cash Bids Table Cash Bids Grid Market Data Market Overview Futures Options Charts Tech. Charts Spread Charts Market Heat Map Stocks Newswire Newswire AgWeb Barchart.com USDA Reports Commentary Ag Market Commentary InsideFutures Weather Local Weather Mesonet Weather Resources Trade Calendar Futures 10...

Measurements:
  - url_length: 44
  - final_url_length: 44
  - final_hostname: www.dacomacoop.com
  - hostname_label_count: 3
  - redirect_count: 0
  - form_count: 1
  - credential_form_count: 1
  - password_input_count: 0
  - hidden_input_count: 1
  - form_action_relationship_counts: {'same_registrable_domain': 1}
  - anchor_count: 56
  - external_anchor_count: 12
  - external_anchor_ratio: 0.2143
  - null_anchor_count: 3
  - inter

In [ ]:
def generate_explanation(row):
    """Build a detailed, evidence-specific explanation from the row's actual data."""
    feats = row['features']
    meas = row['measurements']
    inp = row['input']
    label = row['label']

    # Categorize signals
    high_susp = [f for f in feats if f.get('severity') == 'high' and f.get('direction') == 'suspicious']
    med_susp  = [f for f in feats if f.get('severity') == 'medium' and f.get('direction') == 'suspicious']
    low_susp  = [f for f in feats if f.get('severity') == 'low' and f.get('direction') == 'suspicious']
    safe_sigs = [f for f in feats if f.get('direction') == 'safe']
    uncertain = [f for f in feats if f.get('direction') == 'uncertain']

    hostname = meas.get('final_hostname', 'unknown')
    title = inp.get('title', '')
    has_cred_form = meas.get('credential_form_count', 0) > 0
    has_password = meas.get('password_input_count', 0) > 0
    redirect_count = meas.get('redirect_count', 0)
    nav_links = meas.get('internal_nav_candidate_count', 0)
    anchor_count = meas.get('anchor_count', 0)

    parts = []

    if label == 'phishing':
        # Use actual signal statements for phishing reasoning
        if high_susp:
            stmts = [f['statement'] for f in high_susp[:2]]
            parts.append(f"Critical evidence: {' '.join(stmts)}")

        if med_susp:
            stmts = [f['statement'] for f in med_susp[:2]]
            parts.append(f"Supporting evidence: {' '.join(stmts)}")

        if has_cred_form and has_password and not high_susp:
            parts.append(
                f"The page hosts a credential form with password input on {hostname}, "
                f"which is not a recognized domain for the claimed service."
            )

        if redirect_count > 0:
            parts.append(f"The URL goes through {redirect_count} redirect(s), which can indicate cloaking.")

        if safe_sigs and len(safe_sigs) <= 2:
            parts.append(
                f"While {len(safe_sigs)} safe signal(s) are present, "
                f"they are outweighed by the suspicious indicators."
            )

        if not parts:
            # Fallback for cases with few/no signals
            if not feats:
                parts.append(
                    f"Despite no extracted signals, the URL structure and hosting on {hostname} "
                    f"are inconsistent with the claimed service."
                )
            else:
                stmts = [f['statement'] for f in feats[:2]]
                parts.append(f"The combination of signals suggests phishing: {' '.join(stmts)}")

    else:  # benign
        if safe_sigs:
            stmts = [f['statement'] for f in safe_sigs[:2]]
            parts.append(f"Positive indicators: {' '.join(stmts)}")

        if nav_links > 5:
            parts.append(
                f"The page has {nav_links} functional internal navigation links, "
                f"typical of a legitimate website."
            )

        if has_cred_form and has_password:
            parts.append(
                f"Although a login form is present on {hostname}, "
                f"the overall site structure and safe signals confirm legitimacy."
            )

        if high_susp:
            parts.append(
                f"Despite {len(high_susp)} high-severity flag(s), "
                f"the domain and site structure are consistent with a genuine service."
            )

        if not parts:
            if not feats:
                parts.append(
                    f"No suspicious signals detected. The site on {hostname} "
                    f"shows a normal page structure with {anchor_count} links."
                )
            elif not high_susp and not med_susp:
                parts.append(
                    f"Only {len(low_susp)} low-severity signal(s) detected on {hostname}, "
                    f"which is normal for legitimate websites."
                )
            else:
                parts.append(f"The site on {hostname} displays characteristics of a legitimate website.")

    explanation = " ".join(parts)
    return f"Explanation: {explanation}\nPrediction: {label}"


# Build messages
def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_input(row)},
        {"role": "assistant", "content": generate_explanation(row)},
    ]

train_df['messages'] = train_df.apply(build_messages, axis=1)

# Show a few examples
for i in range(3):
    print(f"--- Example {i+1} (label: {train_df.iloc[i]['label']}) ---")
    print(train_df.iloc[i]['messages'][-1]['content'])
    print()

--- Example 1 (label: benign) ---
Explanation: The page has 35 functional internal navigation links, typical of a legitimate website.
Prediction: benign

--- Example 2 (label: phishing) ---
Explanation: Supporting evidence: Form asks for credential-related information: email.
Prediction: phishing

--- Example 3 (label: benign) ---
Explanation: The site on www.discoveryplus.in displays characteristics of a legitimate website.
Prediction: benign



In [ ]:
import re
import os

# Suppress tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def predict(row, model, tokenizer, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_input(row)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

def parse_prediction(response):
    """Extract phishing or benign from model output."""
    match = re.search(r"prediction\s*:\s*(phishing|benign)", response, re.IGNORECASE)
    if match:
        return match.group(1).lower()
    text = response.lower()
    if "phishing" in text and "benign" not in text:
        return "phishing"
    if "benign" in text and "phishing" not in text:
        return "benign"
    return "unknown"

def evaluate_on_test(model, tokenizer, test_df, desc="Model"):
    """Run predictions on test set and print metrics."""
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    import torch

    preds = []
    responses = []
    total = len(test_df)

    for i in range(total):
        row = test_df.iloc[i]
        resp = predict(row, model, tokenizer)
        pred = parse_prediction(resp)
        preds.append(pred)
        responses.append(resp)

        # Simple progress: print every 200 examples
        if (i + 1) % 200 == 0 or (i + 1) == total:
            print(f"  {desc}: {i+1}/{total} done")

    y_true = test_df['label'].tolist()

    print(f"\n{desc} Results:")
    print(f"Accuracy: {accuracy_score(y_true, preds):.4f}")
    print(classification_report(y_true, preds, labels=['phishing', 'benign']))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, preds, labels=['phishing', 'benign']))

    return preds, responses

In [9]:
import torch
import gc

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print("Running 3B baseline on test set...")
baseline_3b_preds, baseline_3b_responses = evaluate_on_test(model, tokenizer, test_df, "3B Baseline")

test_df['baseline_3b_pred'] = baseline_3b_preds

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("\nMemory cleared.")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Running 3B baseline on test set...
  3B Baseline: 200/2000 done
  3B Baseline: 400/2000 done
  3B Baseline: 600/2000 done
  3B Baseline: 800/2000 done
  3B Baseline: 1000/2000 done
  3B Baseline: 1200/2000 done
  3B Baseline: 1400/2000 done
  3B Baseline: 1600/2000 done
  3B Baseline: 1800/2000 done
  3B Baseline: 2000/2000 done

3B Baseline Results:
Accuracy: 0.6625
              precision    recall  f1-score   support

    phishing       0.66      0.80      0.72      1000
      benign       0.77      0.53      0.62      1000

   micro avg       0.70      0.66      0.68      2000
   macro avg       0.71      0.66      0.67      2000
weighted avg       0.71      0.66      0.67      2000

Confusion matrix:
[[800 160]
 [418 525]]

Memory cleared.


In [10]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print("Running 8B baseline on test set...")
baseline_8b_preds, baseline_8b_responses = evaluate_on_test(model, tokenizer, test_df, "8B Baseline")

test_df['baseline_8b_pred'] = baseline_8b_preds

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("\nMemory cleared.")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Running 8B baseline on test set...
  8B Baseline: 200/2000 done
  8B Baseline: 400/2000 done
  8B Baseline: 600/2000 done
  8B Baseline: 800/2000 done
  8B Baseline: 1000/2000 done
  8B Baseline: 1200/2000 done
  8B Baseline: 1400/2000 done
  8B Baseline: 1600/2000 done
  8B Baseline: 1800/2000 done
  8B Baseline: 2000/2000 done

8B Baseline Results:
Accuracy: 0.6795
              precision    recall  f1-score   support

    phishing       0.68      0.68      0.68      1000
      benign       0.68      0.68      0.68      1000

   micro avg       0.68      0.68      0.68      2000
   macro avg       0.68      0.68      0.68      2000
weighted avg       0.68      0.68      0.68      2000

Confusion matrix:
[[683 313]
 [321 676]]

Memory cleared.


In [11]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [to_text(m) for m in train_df['messages']]
train_ds = Dataset.from_dict({"text": train_texts})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=4096,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs_3b",
        report_to="none",
    ),
)

print("Fine-tuning 3B (3 epochs, 8k examples)...")
trainer.train()
print("Done!")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Fine-tuning 3B (3 epochs, 8k examples)...
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
100,1.402000
200,0.559293
300,0.490791
400,0.473918
500,0.474907
600,0.459911
700,0.445267
800,0.428052
900,0.449981
1000,0.423924


Done!


In [12]:
FastLanguageModel.for_inference(model)

print("Evaluating fine-tuned 3B...")
ft_3b_preds, ft_3b_responses = evaluate_on_test(model, tokenizer, test_df, "3B Fine-tuned")

test_df['ft_3b_pred'] = ft_3b_preds
test_df['ft_3b_response'] = ft_3b_responses

model.save_pretrained("lora_3b")
tokenizer.save_pretrained("lora_3b")

del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()
print("\nMemory cleared.")

Evaluating fine-tuned 3B...
  3B Fine-tuned: 200/2000 done
  3B Fine-tuned: 400/2000 done
  3B Fine-tuned: 600/2000 done
  3B Fine-tuned: 800/2000 done
  3B Fine-tuned: 1000/2000 done
  3B Fine-tuned: 1200/2000 done
  3B Fine-tuned: 1400/2000 done
  3B Fine-tuned: 1600/2000 done
  3B Fine-tuned: 1800/2000 done
  3B Fine-tuned: 2000/2000 done

3B Fine-tuned Results:
Accuracy: 0.9620
              precision    recall  f1-score   support

    phishing       0.95      0.97      0.96      1000
      benign       0.97      0.95      0.96      1000

    accuracy                           0.96      2000
   macro avg       0.96      0.96      0.96      2000
weighted avg       0.96      0.96      0.96      2000

Confusion matrix:
[[974  26]
 [ 50 950]]

Memory cleared.


In [13]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [to_text(m) for m in train_df['messages']]
train_ds = Dataset.from_dict({"text": train_texts})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=4096,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs_8b",
        report_to="none",
    ),
)

print("Fine-tuning 8B (3 epochs, 8k examples)...")
trainer.train()
print("Done!")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8000 [00:00<?, ? examples/s]

Fine-tuning 8B (3 epochs, 8k examples)...
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
100,1.171382
200,0.474770
300,0.423007
400,0.408253
500,0.413053
600,0.395741
700,0.385300
800,0.371788
900,0.390908
1000,0.365160


Step,Training Loss
100,1.171382
200,0.474770
300,0.423007
400,0.408253
500,0.413053
600,0.395741
700,0.385300
800,0.371788
900,0.390908
1000,0.365160


Done!


In [14]:
FastLanguageModel.for_inference(model)

print("Evaluating fine-tuned 8B...")
ft_8b_preds, ft_8b_responses = evaluate_on_test(model, tokenizer, test_df, "8B Fine-tuned")

test_df['ft_8b_pred'] = ft_8b_preds
test_df['ft_8b_response'] = ft_8b_responses

model.save_pretrained("lora_8b")
tokenizer.save_pretrained("lora_8b")

del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()
print("\nMemory cleared.")

Evaluating fine-tuned 8B...
  8B Fine-tuned: 200/2000 done
  8B Fine-tuned: 400/2000 done
  8B Fine-tuned: 600/2000 done
  8B Fine-tuned: 800/2000 done
  8B Fine-tuned: 1000/2000 done
  8B Fine-tuned: 1200/2000 done
  8B Fine-tuned: 1400/2000 done
  8B Fine-tuned: 1600/2000 done
  8B Fine-tuned: 1800/2000 done
  8B Fine-tuned: 2000/2000 done

8B Fine-tuned Results:
Accuracy: 0.9775
              precision    recall  f1-score   support

    phishing       0.98      0.98      0.98      1000
      benign       0.98      0.98      0.98      1000

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000

Confusion matrix:
[[979  21]
 [ 24 976]]

Memory cleared.


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def get_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=['phishing', 'benign'], average='macro', zero_division=0
    )
    return acc, p, r, f

y_true = test_df['label'].tolist()

results = []
for col, name in [
    ('baseline_3b_pred', 'Llama 3.2 3B (baseline)'),
    ('ft_3b_pred',       'Llama 3.2 3B (fine-tuned)'),
    ('baseline_8b_pred', 'Llama 3.1 8B (baseline)'),
    ('ft_8b_pred',       'Llama 3.1 8B (fine-tuned)'),
]:
    if col in test_df.columns:
        acc, p, r, f = get_metrics(y_true, test_df[col].tolist())
        results.append({'Model': name, 'Accuracy': f"{acc:.3f}", 'Precision': f"{p:.3f}",
                        'Recall': f"{r:.3f}", 'F1': f"{f:.3f}"})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv("final_comparison.csv", index=False)

                    Model Accuracy Precision Recall    F1
  Llama 3.2 3B (baseline)    0.662     0.712  0.663 0.672
Llama 3.2 3B (fine-tuned)    0.962     0.962  0.962 0.962
  Llama 3.1 8B (baseline)    0.679     0.682  0.679 0.681
Llama 3.1 8B (fine-tuned)    0.978     0.978  0.978 0.977


In [16]:
if 'ft_8b_pred' in test_df.columns:
    pred_col, resp_col = 'ft_8b_pred', 'ft_8b_response'
else:
    pred_col, resp_col = 'ft_3b_pred', 'ft_3b_response'

correct = test_df[test_df[pred_col] == test_df['label']].head(3)
wrong   = test_df[test_df[pred_col] != test_df['label']].head(3)

print("=== CORRECT PREDICTIONS ===\n")
for _, row in correct.iterrows():
    print(f"URL: {row['input'].get('url','')[:80]}")
    print(f"True: {row['label']} | Predicted: {row[pred_col]}")
    print(f"Response: {row[resp_col][:400]}")
    print("-" * 80)

print("\n=== INCORRECT PREDICTIONS ===\n")
for _, row in wrong.iterrows():
    print(f"URL: {row['input'].get('url','')[:80]}")
    print(f"True: {row['label']} | Predicted: {row[pred_col]}")
    print(f"Response: {row[resp_col][:400]}")
    print("-" * 80)

=== CORRECT PREDICTIONS ===

URL: https://www.lazada.co.id/helpcenter/
True: benign | Predicted: benign
Response: Explanation: The page has 93 functional internal navigation links, typical of a legitimate website.
Prediction: benign
--------------------------------------------------------------------------------
URL: https://rbfcu-star.azurewebsites.net/(S(pqdctldyyoe55plqpnzgk45i))/Main/Login
True: phishing | Predicted: phishing
Response: Explanation: Critical evidence: Page contains 1 password field(s). Supporting evidence: Form asks for credential-related information: login, password, username. Form uses an empty or blank action.
Prediction: phishing
--------------------------------------------------------------------------------
URL: https://www.pianetadesign.it/author/federico-luperi
True: benign | Predicted: benign
Response: Explanation: The page has 44 functional internal navigation links, typical of a legitimate website.
Prediction: benign
--------------------------------------

In [17]:
from google.colab import files
files.download("final_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>